# CFO Step-by-Step: Walking Through the Algorithm

This notebook walks through **every step** of CFO (Algorithm 1) in separate cells, so you can inspect intermediate results, modify parameters between rounds, and understand the algorithm's behavior.

## Algorithm 1: Constrained Flow Optimization

```
For k = 1, ..., K:
  Step 1: Form augmented objective f_k(x) = r(x) - (rho_k/2) * [max(0, c(x) - B - lambda_k/rho_k)]^2
  Step 2: pi_k = FineTuningSolver(f_k, pi_pre)     # Adjoint Matching inner loop (N iterations)
  Step 3: Compute empirical constraint gap G_k
  Step 4: Update lambda_{k+1}
  Step 5: Update rho_{k+1}
```

In [ ]:
import sys, os
from pathlib import Path
os.chdir(Path(__file__).resolve().parent.parent if "__file__" in dir() else Path.cwd().parent)
sys.path.insert(0, str(Path.cwd() / "src"))

import copy
import os.path as osp
import torch
import torch.nn as nn
import dgl
from omegaconf import OmegaConf

import flowmol
from cfo import AugmentedLagrangian, AugmentedReward, set_seed
from finetuning_solver.adjoint_matching import AdjointMatchingFinetuningTrainerFlowMol
from regressor import GNN, EGNN
from utils.sampling import sampling

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## Setup: Load Config, Models, and Initialize CFO

In [ ]:
def load_regressor(rc_config, device):
    K_x, K_a, K_c, K_e = 3, 10, 6, 5
    path = osp.join("pretrained_models", str(rc_config.fn), str(rc_config.model_type), str(rc_config.date), "best_model.pt")
    state = torch.load(path, map_location=device)
    if rc_config.model_type == "gnn":
        model = GNN(property=state["config"]["property"], node_feats=K_a+K_c+K_x, edge_feats=K_e,
                     hidden_dim=state["config"]["hidden_dim"], depth=state["config"]["depth"])
    elif rc_config.model_type == "egnn":
        model = EGNN(property=state["config"]["property"], num_atom_types=K_a, num_charge_classes=K_c,
                      num_bond_types=K_e, hidden_dim=state["config"]["hidden_dim"], depth=state["config"]["depth"])
    model.load_state_dict(state["model_state"])
    return model

# Config
config = OmegaConf.load("configs/cfo.yaml")
config.seed = 42
config.total_steps = 4
config.augmented_lagrangian.lagrangian_updates = 2
config.adjoint_matching.sampling.num_samples = 4
config.adjoint_matching.batch_size = 2
config.reward_sampling.num_samples = 8
config.augmented_lagrangian.sampling.num_samples = 8
set_seed(config.seed)

# Derived parameters
K = config.augmented_lagrangian.lagrangian_updates  # ALM rounds
N = config.total_steps // K                          # AM iterations per round
reward_lambda = config.reward_lambda
config.adjoint_matching.reward_lambda = reward_lambda
n_atoms = config.get("n_atoms", None)
min_num_atoms = config.get("min_num_atoms", None)
max_num_atoms = config.get("max_num_atoms", None)
config.adjoint_matching.sampling.n_atoms = n_atoms
config.adjoint_matching.sampling.min_num_atoms = min_num_atoms
config.adjoint_matching.sampling.max_num_atoms = max_num_atoms

# Models
base_model = flowmol.load_pretrained(config.flow_model).to(device)
fine_model = copy.deepcopy(base_model)
reward_model = load_regressor(config.reward, device)
constraint_model = load_regressor(config.constraint, device)

# CFO components
augmented_reward = AugmentedReward(
    reward_fn=reward_model, constraint_fn=constraint_model,
    alpha=reward_lambda, bound=config.constraint.bound, device=device,
)
augmented_reward.set_lambda_rho(lambda_=0.0, rho_=config.augmented_lagrangian.rho_init)

alm = AugmentedLagrangian(
    config=config.augmented_lagrangian, constraint_fn=constraint_model,
    bound=config.constraint.bound, device=device,
)

def sample(model):
    return sampling(config.reward_sampling, model, device=device,
                    n_atoms=n_atoms, min_num_atoms=min_num_atoms, max_num_atoms=max_num_atoms)

print(f"CFO: K={K} ALM rounds, N={N} AM iterations per round")
print(f"Reward: {config.reward.fn}, Constraint: {config.constraint.fn} <= {config.constraint.bound}")

## Initial Evaluation (before any fine-tuning)

In [ ]:
full_stats = []
al_stats = []

# Sample from the pretrained model
dgl_mols, rd_mols = sample(copy.deepcopy(fine_model))
dgl_mols_batched = dgl.batch(dgl_mols).to(device)
_ = augmented_reward(dgl_mols_batched)
initial_stats = augmented_reward.get_statistics()
full_stats.append(initial_stats)
_ = alm.expected_constraint(dgl_mols)

print(f"Initial reward:     {initial_stats['reward']:.4f}")
print(f"Initial constraint: {initial_stats['constraint']:.4f} (bound: {config.constraint.bound})")
print(f"Initial violations: {initial_stats['constraint_violations']:.4f}")
del dgl_mols_batched

## ALM Outer Loop

Run this cell **K times** (or re-run to do more rounds). Each execution performs one full ALM round:
1. Get current lambda/rho and update the augmented objective
2. Run N iterations of Adjoint Matching (FineTuningSolver)
3. Update lambda/rho based on constraint satisfaction

In [ ]:
for k in range(1, K + 1):
    print(f"\n{'='*60}")
    print(f"ALM Round {k}/{K}")
    print(f"{'='*60}")

    # --- Steps 3-5 from previous round: get current lambda, rho ---
    lambda_, rho_ = alm.get_current_lambda_rho()
    al_stats.append(alm.get_statistics())
    print(f"  lambda = {lambda_:.4f}, rho = {rho_:.4f}")

    # --- Step 1: Update augmented objective f_k ---
    augmented_reward.set_lambda_rho(lambda_, rho_)

    # --- Step 2: FineTuningSolver (Adjoint Matching) ---
    trainer = AdjointMatchingFinetuningTrainerFlowMol(
        config=config.adjoint_matching,
        model=copy.deepcopy(fine_model),
        base_model=copy.deepcopy(base_model),
        grad_reward_fn=augmented_reward.grad_augmented_reward_fn,
        device=device,
        verbose=True,
    )

    for i in range(1, N + 1):
        dataset, avg_adj_0_norm = trainer.generate_dataset()
        if dataset is None:
            print(f"  AM iteration {i}: dataset is None, skipping")
            continue
        loss, grad_norm = trainer.finetune(dataset)
        del dataset

        # Evaluate
        tmp_model = copy.deepcopy(trainer.fine_model)
        dgl_mols, rd_mols = sample(tmp_model)
        del tmp_model
        dgl_mols_batched = dgl.batch(dgl_mols).to(device)
        _ = augmented_reward(dgl_mols_batched)
        stats = augmented_reward.get_statistics()
        stats["loss"] = loss
        stats["grad_norm"] = grad_norm
        stats["adj_0_norm"] = avg_adj_0_norm
        full_stats.append(stats)
        del dgl_mols_batched

        print(f"  AM iteration {i}: reward={stats['reward']:.4f}, "
              f"constraint={stats['constraint']:.4f}, violations={stats['constraint_violations']:.4f}")

    fine_model = copy.deepcopy(trainer.fine_model)

    # --- Steps 3-5: Update lambda, rho ---
    dgl_mols, _ = sample(copy.deepcopy(fine_model))
    alm.update_lambda_rho(dgl_mols)
    del dgl_mols, trainer

    new_lambda, new_rho = alm.get_current_lambda_rho()
    print(f"\n  Updated: lambda = {new_lambda:.4f}, rho = {new_rho:.4f}")

print(f"\n{'='*60}")
print(f"CFO complete after {K} rounds")
print(f"Final reward: {full_stats[-1]['reward']:.4f}")
print(f"Final constraint: {full_stats[-1]['constraint']:.4f}")
print(f"{'='*60}")

## Visualize Results

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.DataFrame.from_records(full_stats)
df_alm = pd.DataFrame.from_dict(al_stats)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

axes[0,0].plot(df["reward"], marker="o")
axes[0,0].set_title("Reward"); axes[0,0].set_xlabel("Step")

axes[0,1].plot(df["constraint"], marker="o", color="tab:orange")
axes[0,1].axhline(y=config.constraint.bound, color="red", linestyle="--", label=f"B={config.constraint.bound}")
axes[0,1].set_title("Constraint"); axes[0,1].set_xlabel("Step"); axes[0,1].legend()

axes[0,2].plot(df["constraint_violations"], marker="o", color="tab:green")
axes[0,2].set_title("Constraint Violations"); axes[0,2].set_xlabel("Step")

axes[1,0].plot(df_alm["alm/lambda"], marker="s", color="tab:purple")
axes[1,0].set_title("Lambda (dual variable)"); axes[1,0].set_xlabel("ALM Round")

axes[1,1].plot(df_alm["alm/rho"], marker="s", color="tab:brown")
axes[1,1].set_title("Rho (penalty)"); axes[1,1].set_xlabel("ALM Round")

axes[1,2].plot(df_alm["alm/expected_constraint"], marker="s", color="tab:red")
axes[1,2].axhline(y=config.constraint.bound, color="red", linestyle="--")
axes[1,2].set_title("Expected Constraint"); axes[1,2].set_xlabel("ALM Round")

plt.tight_layout()
plt.show()